# 图像转换

## 介绍

之前的教程专注于从图像中提取标签或掩码。图像转换采用不同的方法：不是生成标签，而是通过学习从一个图像域到另一个图像域的映射来生成新图像，同时保留空间结构。

图像到图像转换涵盖许多遥感任务，包括传感器转换（如SAR到光学）、时间间隙填充、着色和合成数据生成。它们都有一个共同的结构：模型将输入图像$x$从域$A$映射到域$B$的输出图像$y$，其中域在分辨率、光谱内容、传感器类型或视觉风格上不同。

其中，超分辨率是最实际重要的任务之一。像Sentinel-2这样的任务每五天提供10米分辨率的免费全球覆盖，而商业卫星以更高的成本实现亚米级分辨率。超分辨率使用深度学习来增强免费可用图像的空间分辨率，部分弥合这一差距。

本教程重点介绍使用LDSR-S2潜在扩散模型将Sentinel-2图像从10米分辨率增强到2.5米分辨率。您将下载Sentinel-2场景，在单个补丁和更大区域上运行超分辨率，可视化增强输出，并计算不确定性地图。

## 学习目标

在本教程结束时，您将能够：

- 解释什么是图像到图像转换，并识别其在遥感中的关键应用
- 描述超分辨率问题及其对卫星图像分析的重要性
- 解释潜在扩散模型如何通过编码、去噪和解码执行超分辨率
- 下载并检查用于超分辨率处理的Sentinel-2多光谱图像
- 运行单补丁超分辨率，在每个空间维度上将128x128像素区域增强四倍
- 比较低分辨率输入和超分辨率输出以评估增强质量
- 使用随机前向传播计算和解释逐像素不确定性地图
- 使用带重叠混合的切片推理处理大于单个补丁的区域

## 图像转换基础

### 什么是图像到图像转换？

图像到图像转换将图像从一个域转换到另一个域，生成新图像而不是类别标签。输入和输出共享空间结构，但在某些属性上不同，如分辨率、光谱内容或传感器模态。

像Pix2Pix这样的配对转换框架使用条件GAN学习对齐图像对之间的映射，而CycleGAN将其扩展到未配对设置。在遥感中，这些架构已适用于SAR到光学转换、跨传感器协调和地图生成。

对于地理空间应用，最重要的图像转换类别包括：

1. **超分辨率**：增强空间分辨率，例如将10米的Sentinel-2像素转换为2.5米像素
2. **传感器转换**：在成像模态之间转换，例如从SAR数据生成光学图像
3. **时间合成**：为没有直接观测的日期生成图像，填补由云或重访计划造成的空白
4. **光谱增强**：向缺乏光谱波段的图像添加光谱波段，例如从RGB输入预测多光谱内容
5. **合成数据生成**：为没有标注数据的区域创建真实的训练图像

### 遥感超分辨率

超分辨率（SR）从一个或多个低分辨率输入重建高分辨率图像，产生比传感器原生提供的更精细的空间细节。如果模型能够可靠地将免费可用的10米图像增强到接近2.5米质量，它将在全球范围内解锁详细的空间分析，而无需额外的数据成本。

单图像超分辨率（SISR）是最具挑战性的变体，因为模型必须推断未直接观察到的精细尺度细节。这个问题是不适定的：许多高分辨率图像在降采样时可能产生相同的低分辨率观察。早期基于CNN的方法最小化像素级误差，但产生过于平滑的输出。生成模型（GAN和最近的扩散模型）通过学习生成清晰、真实的输出来解决这个问题。

### 用于超分辨率的潜在扩散模型

扩散模型通过逐步向训练图像添加噪声（正向阶段），然后学习逐步逆转这种损坏（反向阶段）来生成图像。在低分辨率输入上调节反向过程使模型能够生成具有合理精细尺度细节的一致高分辨率输出。

潜在扩散模型（LDM）通过在压缩表示空间中工作而不是直接在全分辨率像素上工作来降低计算成本。编码器将图像映射到低维潜在空间，扩散在那里操作，解码器将结果映射回像素空间。

LDSR-S2模型将潜在扩散应用于Sentinel-2超分辨率。它在四个光谱波段（红、绿、蓝和近红外）上操作，并执行4倍空间上采样，将10米像素转换为2.5米像素。每个128x128补丁的工作流程是：

1. **编码**：低分辨率补丁（128x128，4波段）被压缩成潜在表示
2. **去噪**：扩散过程在潜在空间中运行指定数量的采样步骤，以低分辨率输入为条件
3. **解码**：去噪后的潜在表示以4倍分辨率（512x512，4波段）解码回像素空间

扩散模型的一个关键优势是不确定性量化。使用不同的随机种子多次运行模型会产生略有不同的输出，这些变化的标准差提供了逐像素不确定性估计。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install "geoai-py[extra]"

## 导入库

In [ ]:
import geoai
import numpy as np
import rasterio as rio
from matplotlib import pyplot as plt

## 下载示例数据

我们使用田纳西州诺克斯维尔上空的Sentinel-2 Level-2A子集。该图像包含四个10米波段（红、绿、蓝和近红外），存储为`uint16`的大气底部（BOA）反射率值，范围为0到10,000。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/S2C-MSIL2A-20250920T162001-Knoxville.tif"
s2_path = geoai.download_file(url)

### 检查输入数据

在处理前检查图像尺寸、坐标参考系统和像素分辨率。

In [ ]:
with rio.open(s2_path) as src:
    print(f"Bands: {src.count}")
    print(f"Size: {src.width} x {src.height}")
    print(f"CRS: {src.crs}")
    print(f"Resolution: {src.res[0]:.2f} m")
    print(f"Dtype: {src.dtypes[0]}")

```text
波段：4
尺寸：2874 x 1623
坐标参考系统：EPSG:3857
分辨率：10.00米
数据类型：uint16
```

图像有四个10米分辨率的波段。`uint16`数据类型和0到10,000的值范围是Sentinel-2 L2A产品的标准，其中1,000对应0.1（10%）的表面反射率。

## 可视化输入RGB合成

使用红、绿、蓝波段显示真彩色合成，采用基于百分位数的对比度拉伸，将第2和第98百分位值映射到0和1。

In [ ]:
with rio.open(s2_path) as src:
    rgb = src.read([1, 2, 3]).astype(np.float32)

for i in range(3):
    band = rgb[i]
    p2, p98 = np.percentile(band, (2, 98))
    rgb[i] = (band - p2) / (p98 - p2)
rgb = np.clip(rgb, 0, 1)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(rgb.transpose(1, 2, 0))
ax.set_title("Sentinel-2 RGB Composite (10 m)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 单补丁超分辨率

LDSR-S2模型处理128x128像素的补丁。`window`参数以像素坐标指定`(行偏移, 列偏移, 高度, 宽度)`。模型对补丁进行编码，运行去噪扩散，并以4倍分辨率解码结果，在2.5米处产生512x512输出。

In [ ]:
sr_output = "sr_output.tif"
sr_image, _ = geoai.super_resolution(
    input_lr_path=s2_path,
    output_sr_path=sr_output,
    rgb_nir_bands=[1, 2, 3, 4],
    window=(700, 1300, 128, 128),
    sampling_steps=100,
)

In [ ]:
print(f"Input shape:  (4, 128, 128) at 10 m")
print(f"Output shape: {sr_image.shape} at 2.5 m")

```text
输入形状：(4, 128, 128)，分辨率10米
输出形状：(4, 512, 512)，分辨率2.5米
```

### 比较低分辨率和超分辨率

使用`plot_sr_comparison`显示低分辨率输入和超分辨率输出的并排RGB比较。

In [ ]:
geoai.plot_sr_comparison(s2_path, sr_output, bands=[1, 2, 3])
plt.show()

验证输出GeoTIFF具有正确的空间参考和2.5米像素大小。

In [ ]:
with rio.open(sr_output) as src:
    print(f"SR Bands: {src.count}")
    print(f"SR Size: {src.width} x {src.height}")
    print(f"SR CRS: {src.crs}")
    print(f"SR Resolution: {src.res[0]:.2f} m")

```text
SR 波段：4
超分辨率尺寸：512 x 512
SR 坐标参考系统：EPSG:3857
超分辨率分辨率：2.50米
```

## 不确定性估计

基于扩散的超分辨率可以量化逐像素不确定性。使用不同的随机种子多次运行模型会产生略有不同的输出，这些变化的标准差产生不确定性地图。

设置`compute_uncertainty=True`并使用`n_variations`控制运行多少次随机前向传播。更多的变化产生更平滑的估计，但增加计算时间。

In [ ]:
sr_unc_output = "sr_with_uncertainty.tif"
unc_output = "uncertainty.tif"
sr_image2, uncertainty = geoai.super_resolution(
    input_lr_path=s2_path,
    output_sr_path=sr_unc_output,
    output_uncertainty_path=unc_output,
    rgb_nir_bands=[1, 2, 3, 4],
    window=(700, 1300, 128, 128),
    compute_uncertainty=True,
    n_variations=5,
    sampling_steps=100,
)

### 可视化不确定性地图

使用伪彩色可视化显示不确定性地图。红/黄色区域表示较高的不确定性（跨种子的输出变化较大），而绿色区域表示较高的置信度。

高不确定性通常出现在尖锐边界和具有复杂精细尺度纹理的区域。像开阔水域或裸露土壤这样的均匀区域通常显示低不确定性。

In [ ]:
geoai.plot_sr_uncertainty(unc_output)
plt.show()

## 大区域的切片推理

对于大于128x128像素的区域，该函数自动将输入切片成重叠的补丁，独立处理每个补丁，并使用线性混合缝合结果以防止可见的接缝。

`patch_size`参数设置切片大小，`overlap`控制相邻补丁共享多少像素。较大的重叠产生更平滑的过渡，但增加计算时间。

In [ ]:
sr_large = "sr_large.tif"
sr_large_img, _ = geoai.super_resolution(
    input_lr_path=s2_path,
    output_sr_path=sr_large,
    rgb_nir_bands=[1, 2, 3, 4],
    window=(700, 1300, 256, 256),
    patch_size=128,
    overlap=16,
    sampling_steps=100,
)

In [ ]:
print(f"Input shape:  (4, 256, 256) at 10 m")
print(f"Output shape: {sr_large_img.shape} at 2.5 m")

```text
Input shape:  (4, 256, 256) at 10 m
Output shape: (4, 1024, 1024) at 2.5 m
```

### 比较大区域的结果

比较256x256区域的低分辨率输入和缝合的超分辨率输出。

In [ ]:
geoai.plot_sr_comparison(s2_path, sr_large, bands=[1, 2, 3])
plt.show()

### 交互式分割地图比较

使用分割地图交互式比较超分辨率输出与高分辨率底图图像。请注意，底图可能来自不同的采集日期，因此不匹配可能反映时间差异以及模型误差。

In [ ]:
geoai.create_split_map(
    left_layer=sr_large, right_layer="Esri.WorldImagery", left_args={"vmax": 0.3}
)

## 限制和注意事项

超分辨率模型增强视觉细节，但它们不会恢复真实的地面信息。精细尺度特征是统计预测，而不是直接观察。建筑轮廓可能看起来更清晰，但与真实轮廓不匹配，道路边缘可能在不存在的地方被幻觉出来。

这对定量分析最为重要。从超分辨率输出测量建筑面积或描绘田地边界可能引入系统误差。不确定性地图有助于标记低置信度区域，但低不确定性不能保证正确性。

超分辨率模型还带有训练数据的偏差，可能无法保留植被指数计算或变化检测所需的辐射关系。超分辨率最好被视为视觉增强工具，而不是真正高分辨率图像的替代品。

## 关键要点

1. **图像转换生成新图像而不是标签**，在保留空间结构的同时学习图像域之间的映射。

2. **超分辨率通过从较粗输入推断合理的精细尺度结构，增强了超出传感器原生能力的空间细节**。

3. **潜在扩散模型通过编码到潜在空间、以低分辨率输入为条件去噪、并解码回像素空间来执行超分辨率。**

4. **在潜在空间中工作降低了计算成本**，同时在标准硬件上保持高质量结果。

5. **LDSR-S2模型在Sentinel-2图像上执行4倍超分辨率**，在128x128补丁中处理四个波段，以2.5米分辨率产生512x512输出。

6. **不确定性地图通过计算多个随机前向传播的标准差来量化逐像素模型置信度**。

7. **带重叠混合的切片推理通过处理重叠补丁并无缝混合它们来处理任意大的场景**。

8. **Output GeoTIFFs preserve full georeferencing**, with the geotransform automatically adjusted for the finer pixel size.